# Cat bond fund analysis

This analyzes a diverse range of cat bond funds with publicly available data.

## Sources

Data was gathered from various public sources. Links to sufficient sources are provided in the headers of the raw CSVs ([link](../data/raw/)) for reference and verification.

## Setup

### Imports

In [ ]:
%load_ext autoreload
%autoreload 3

In [ ]:
from analysis import columns as col, paths, regression as reg, selection as sel
from analysis.data import checks as chk, panels as pan, sources as src
from analysis.plotting import figure as fig, plots as viz

sources = src.load(paths.RAW_DATA_DIR)


### Settings


In [ ]:
EXCLUDED_FUNDS = ()
ANCHOR = "2026-08"
HEATMAP_MONTHS = 36
ROLLING_WINDOW = 36

SCREEN = {
    "financing_spread": 0.0075,
    "equity_weight": 0.60,
    "cta_weight": 0.40,
    "min_months": 36,
    "min_cat_t": 2.0,
    "max_control_t": 2.0,
}


## Check raw data

Before processing, compound the raw monthly returns within each calendar year and compare them with the reported annual returns. Both inputs are the unadjusted series from the raw CSVs. Only fund-years with all twelve months are included. The tolerance is derived from the quoting precision in `checks.TOL_PP`; larger gaps point to likely data errors. The plot is shown inline for spot-checking.


In [ ]:
def check_raw(tol: float = chk.TOL_PP) -> None:
    deviations = chk.annual_deviations(
        sources.fund_monthly.data, sources.fund_yearly.data
    )
    summary, flags = chk.summarize(deviations, tol=tol)
    print("Flags above tolerance (pp):")
    display(flags)
    print("Per-fund deviation summary (pp):")
    display(summary.round(3))
    viz.annual_deviations(deviations, tol=tol)


check_raw()


## Process returns

Build the named return panels used for analysis, and persist them.


In [ ]:
panels = pan.build(sources, paths.PROCESSED_DATA_DIR)
print("Processed panels:\n- " + "\n- ".join(panels))


## Explore

In [ ]:
SELECTED_FUNDS = pan.selected_funds(sources, EXCLUDED_FUNDS)
INDEXES = (col.PLENUM_CAT_INDEX, col.ILS_INDEX)
FACTORS = (*INDEXES, col.MSCI_ACWI_INDEX, col.SG_CTA_INDEX)
excess, total, factor_returns, heatmap_columns = pan.comparison_frames(
    panels, SELECTED_FUNDS, INDEXES, FACTORS
)


In [ ]:
def explore() -> None:
    display(pan.coverage(total))
    heatmap, _ = viz.monthly_heatmap(
        excess,
        columns=heatmap_columns,
        months=HEATMAP_MONTHS,
        title="Monthly returns relative to USD cash",
    )
    fig.save(heatmap, paths.FIGURES_DIR / "heatmap.png")
    viz.anchored_growth(total, columns=SELECTED_FUNDS, anchor=ANCHOR)
    viz.drawdowns(total, columns=SELECTED_FUNDS)


explore()

In [ ]:
viz.trailing_returns_heatmap(
    excess, columns=SELECTED_FUNDS, window=12, display_months=HEATMAP_MONTHS
)
viz.calendar_returns_heatmap(excess, columns=SELECTED_FUNDS)
viz.correlation_heatmap(excess, factor_returns, columns=SELECTED_FUNDS)


## Factor exposures

In [ ]:
FACTOR_COLUMNS = {
    col.PLENUM_CAT_EXCESS: col.PLENUM_CAT_INDEX,
    col.ILS_EXCESS: col.ILS_INDEX,
    # col.UST_3Y_EXCESS: col.UST_ZERO_3Y,
    # col.UST_5Y_EXCESS: col.UST_ZERO_5Y,
    # col.UST_10Y_EXCESS: col.UST_ZERO_10Y,
    # col.UST_30Y_EXCESS: col.UST_ZERO_30Y,
    col.MSCI_EXCESS: col.MSCI_ACWI_INDEX,
    # col.SG_TREND_EXCESS: col.SG_TREND_INDEX,
    col.SG_CTA_EXCESS: col.SG_CTA_INDEX,
}


In [ ]:
CAT_REPRESENTATIONS = (col.PLENUM_CAT_EXCESS, col.ILS_EXCESS, col.CAT_PC1_EXCESS)
study = reg.exposures(
    panels,
    SELECTED_FUNDS,
    FACTOR_COLUMNS,
    CAT_REPRESENTATIONS,
    anchor=ANCHOR,
    min_months=SCREEN["min_months"],
    control_min_months=60,
)


In [ ]:
display(
    reg.exposure_table(
        study.cat,
        (col.IMPACT_PER_SD, col.T_STAT),
        sort_by=col.CAT_PC1_EXCESS,
        ascending=False,
    )
)
viz.factor_exposure_heatmap(
    study.cat, title="CAT/ILS representations: exposure per 1 SD"
)
display(reg.control_diagnostics(study.control))
display(reg.exposure_table(study.control, (col.T_STAT,)))

## Exposure stability

In [ ]:
expanding = study.expanding(anchor=ANCHOR, min_obs=SCREEN["min_months"])
rolling = study.rolling(anchor=ANCHOR, window=ROLLING_WINDOW)
print("Expanding")
display(reg.exposure_table(expanding, (col.OBSERVATIONS,)))
viz.factor_exposure_heatmap(
    expanding, sort_by=None, title="Expanding exposure to CAT/ILS PC1"
)
print(f"Rolling {ROLLING_WINDOW}m")
display(reg.exposure_table(rolling, (col.OBSERVATIONS,)))
viz.factor_exposure_heatmap(
    rolling, sort_by=None, title=f"Rolling {ROLLING_WINDOW}m exposure to CAT/ILS PC1"
)

## Selection

In [ ]:
card = sel.scorecard(
    net_returns=sources.fund_monthly.data,
    cash_returns=panels[col.ASSET_CLASS_RETURNS][col.CASH],
    excess_panel=panels[col.EXCESS_RETURNS],
    funds=SELECTED_FUNDS,
    cat_exposures=study.cat,
    control_exposures=study.control,
    target=col.CAT_PC1_EXCESS,
    plenum=col.PLENUM_CAT_EXCESS,
    intelligence=col.ILS_EXCESS,
    **SCREEN,
)
display(sel.scorecard_view(card))
display(sel.scorecard_view(card, decision=col.PASS))